# Task B Evaluation — Recommendation
### Team Greenlight · DSN x BCT LLM Agent Hackathon 3.0

Metrics computed:
- **Hit Rate@5** — did any ground truth item appear in top-5 recommendations?
- **NDCG@5** — normalised discounted cumulative gain at rank 5
- **Cold-start test** — 5 brand new synthetic users

Results saved to `results/task_b_results.json`

In [1]:
import sys, json, random, math
from pathlib import Path
from collections import defaultdict

ROOT = Path("__file__").resolve().parent.parent
sys.path.insert(0, str(ROOT / "task_b"))

import numpy as np
from agent import recommend, recommend_cold_start, _load_data

DATA_DIR    = ROOT / "data"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Imports OK")

C:\Users\Ayoola\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Users\Ayoola\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Imports OK


In [2]:
# Load data
_load_data()
from agent import _RECORDS, _PROFILES

user_records = defaultdict(list)
for r in _RECORDS:
    user_records[r["user_id"]].append(r)

# Pick 20 users with at least 4 reviews (hide last 3 as ground truth)
eligible = [
    uid for uid, recs in user_records.items()
    if uid in _PROFILES and len(recs) >= 4
]
random.seed(42)
test_users = random.sample(eligible, min(20, len(eligible)))
print(f"Test users: {len(test_users)}")
print(f"Total records: {len(_RECORDS):,}")

INFO:agent:Loaded 9,999 records, 1,271 profiles.


Test users: 20
Total records: 9,999


In [3]:
# Helper: NDCG@k
def ndcg_at_k(recommended_names: list, relevant_names: set, k: int) -> float:
    dcg = 0.0
    for i, name in enumerate(recommended_names[:k]):
        if name in relevant_names:
            dcg += 1.0 / math.log2(i + 2)   # i+2 because log2(1)=0
    # Ideal DCG: all relevant items at top positions
    ideal_hits = min(len(relevant_names), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_hits))
    return round(dcg / idcg, 4) if idcg > 0 else 0.0

def hit_rate_at_k(recommended_names: list, relevant_names: set, k: int) -> int:
    return int(any(name in relevant_names for name in recommended_names[:k]))

print("Helpers defined.")

Helpers defined.


In [4]:
# Run main evaluation
K = 5
results = []

for i, uid in enumerate(test_users):
    user_recs = user_records[uid]
    # Hide last 3 as ground truth, use rest as history context
    ground_truth_items = {r["item_name"] for r in user_recs[-3:]}

    print(f"[{i+1}/{len(test_users)}] {uid} | GT items: {len(ground_truth_items)}")

    try:
        output = recommend(user_id=uid, conversation_history=[], top_k=K)
        rec_names = [r.get("item_name", "") for r in output["recommendations"]]

        hit  = hit_rate_at_k(rec_names, ground_truth_items, K)
        ndcg = ndcg_at_k(rec_names, ground_truth_items, K)

        result = {
            "user_id":          uid,
            "hit_at_5":         hit,
            "ndcg_at_5":        ndcg,
            "num_recs":         len(rec_names),
            "is_cold_start":    output["is_cold_start"],
            "user_need_summary": output["user_need_summary"][:100],
            "recommended":      rec_names,
            "ground_truth":     list(ground_truth_items),
        }
        results.append(result)
        print(f"  Hit@{K}: {hit}  NDCG@{K}: {ndcg:.4f}  Recs: {rec_names[:2]}...")

    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({"user_id": uid, "error": str(e)})

print(f"\nCompleted {len(results)} evaluations.")

[1/20] yelp_user_0228 | GT items: 3


* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 8.917377431s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 8
}
]
INFO:agent:Need summary: items matching the user's general taste preferences...
INFO:agent:Loading cached retrieval index...
INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Successfully loaded faiss with AVX2 support.
INFO:agent:Loaded index (sentence-transformers+faiss method).
INFO:sentence_transformers.SentenceTransformer:Load pre

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:agent:Retrieved 50 FAISS candidates (review-text query).
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 54.268468176s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 54
}
]
INFO:agent:Returning 5 recommendations.
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 54.064939049s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google

  Hit@5: 1  NDCG@5: 0.4693  Recs: ['Yelp_Business_2228', 'Yelp_Business_228']...
[2/20] yelp_user_0051 | GT items: 3


...

  Hit@5: 1  NDCG@5: 0.7654  Recs: ["Love Uncle John's", 'Another good book']...
[20/20] yelp_user_0407 | GT items: 3


INFO:agent:Need summary: items matching the user's general taste preferences...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:agent:Retrieved 50 FAISS candidates (review-text query).
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 32.431717174s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 32
}
]
INFO:agent:Returning 5 recommendations.


  Hit@5: 1  NDCG@5: 0.4693  Recs: ['Yelp_Business_2407', 'Yelp_Business_2664']...

Completed 20 evaluations.


In [5]:
# Cold-start evaluation — 5 synthetic new users
print("Running cold-start evaluation...\n")

cold_start_users = [
    {"id": "new_user_1", "category": "Restaurant",
     "want": "spicy Nigerian street food, affordable, large portions"},
    {"id": "new_user_2", "category": "Fiction",
     "want": "gripping thriller with African setting and plot twists"},
    {"id": "new_user_3", "category": "Snacks",
     "want": "healthy low-sugar snacks for office, crunchy and filling"},
    {"id": "new_user_4", "category": "Hotel",
     "want": "budget-friendly hotel with good wifi and clean rooms"},
    {"id": "new_user_5", "category": "Beverages",
     "want": "natural fruit juice or herbal drink with no preservatives"},
]

cold_results = []
for u in cold_start_users:
    print(f"Testing cold-start: {u['id']} → {u['category']}")
    try:
        output = recommend_cold_start(
            item_category=u["category"],
            description_of_what_i_want=u["want"],
            top_k=5
        )
        rec_names = [r.get("item_name", "") for r in output["recommendations"]]
        cold_results.append({
            "user_id":      u["id"],
            "category":     u["category"],
            "want":         u["want"],
            "num_recs":     len(rec_names),
            "is_cold_start": output["is_cold_start"],
            "recommended":  rec_names,
            "need_summary": output["user_need_summary"][:150],
            "status":       "ok",
        })
        print(f"  Recs: {rec_names[:3]}")
    except Exception as e:
        print(f"  ERROR: {e}")
        cold_results.append({"user_id": u["id"], "error": str(e)})

cold_success = sum(1 for r in cold_results if r.get("status") == "ok")
print(f"\nCold-start success rate: {cold_success}/{len(cold_results)}")

Running cold-start evaluation...

Testing cold-start: new_user_1 → Restaurant


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 32.162319377s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 32
}
]


  Recs: ['Yelp_Business_4722', 'Yelp_Business_3261', 'Yelp_Business_3192']
Testing cold-start: new_user_2 → Fiction


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 31.215320601s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 31
}
]


  Recs: ['Yelp_Business_3083', 'Yelp_Business_3993', 'Yelp_Business_4953']
Testing cold-start: new_user_5 → Beverages


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 30.918325202s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 30
}
]


  Recs: ['Great', 'Wonderslim protein fruit drink', 'YUMMY']

Cold-start success rate: 5/5


In [6]:
# Compute aggregate metrics and save
valid = [r for r in results if "hit_at_5" in r]

avg_hit  = np.mean([r["hit_at_5"]  for r in valid])
avg_ndcg = np.mean([r["ndcg_at_5"] for r in valid])

summary = {
    "num_users_evaluated": len(valid),
    "hit_rate_at_5":       round(float(avg_hit),  4),
    "ndcg_at_5":           round(float(avg_ndcg), 4),
    "cold_start_success":  f"{cold_success}/{len(cold_results)}",
    "individual_results":  results,
    "cold_start_results":  cold_results,
}

with open(RESULTS_DIR / "task_b_results.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("=" * 50)
print("TASK B EVALUATION RESULTS")
print("=" * 50)
print(f"  Users evaluated  : {len(valid)}")
print(f"  Hit Rate@5       : {avg_hit:.4f}  ({avg_hit:.1%})")
print(f"  NDCG@5           : {avg_ndcg:.4f}")
print(f"  Cold-start       : {cold_success}/{len(cold_results)} successful")
print(f"\nSaved → results/task_b_results.json")

TASK B EVALUATION RESULTS
  Users evaluated  : 20
  Hit Rate@5       : 0.6500  (65.0%)
  NDCG@5           : 0.2968
  Cold-start       : 5/5 successful

Saved → results/task_b_results.json


In [7]:
# Summary table
print(f"{'User ID':<25} {'Hit@5':>6} {'NDCG@5':>8} {'#Recs':>6}")
print("-" * 50)
for r in valid:
    print(f"{r['user_id']:<25} {r['hit_at_5']:>6} {r['ndcg_at_5']:>8.4f} {r['num_recs']:>6}")
print("-" * 50)
print(f"{'AVERAGE':<25} {avg_hit:>6.4f} {avg_ndcg:>8.4f}")

print("\n" + "=" * 50)
print("COLD-START RESULTS")
print("=" * 50)
for r in cold_results:
    status = "✓" if r.get("status") == "ok" else "✗"
    print(f"  {status} {r['user_id']} [{r.get('category','?')}] → {r.get('num_recs', 0)} recs")

User ID                    Hit@5   NDCG@5  #Recs
--------------------------------------------------
yelp_user_0228                 1   0.4693      5
yelp_user_0051                 1   0.2961      5
yelp_user_0563                 1   0.4693      5
yelp_user_0501                 1   0.1815      5
yelp_user_0457                 0   0.0000      5
yelp_user_0285                 1   0.4693      5
yelp_user_0209                 0   0.0000      5
amazon_AG4MIJDYTNWORGOKC2K2      0   0.0000      5
yelp_user_0178                 0   0.0000      5
gr_AGPOZ62Q3ULTH7RLBK4B        0   0.0000      5
yelp_user_0864                 1   0.4693      5
yelp_user_0065                 0   0.0000      5
yelp_user_0061                 1   0.4693      5
yelp_user_0191                 1   0.4693      5
yelp_user_0447                 0   0.0000      5
yelp_user_0476                 1   0.4693      5
amazon_AHCV2CNCOCG6WECDROOU      1   0.4693      5
yelp_user_0054                 1   0.4693      5
gr_AH6CATODIVP